<a href="https://colab.research.google.com/github/M4XJUNG/AIF-Financial-Time-Series/blob/main/M4_N_BEATS_Time_o1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# (추가) GPU에서 “가능한 한” 결과 고정시키는 셀 (원하면 맨 위에 1회만)
import os, random
import numpy as np
import torch

seed = 42
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# 아래는 일부 연산에서 에러 날 수 있음(그럼 주석 처리)
# torch.use_deterministic_algorithms(True)


In [ ]:
# 셀 1) 프로젝트 폴더 만들기 + M4 Finance Daily를 “long format”으로 준비
import os, sys, subprocess
from pathlib import Path
import pandas as pd
import numpy as np

PROJ = Path("/content/m4_nbeats_timeo1")
for p in ["data", "artifacts", "logs"]:
    os.makedirs(PROJ / p, exist_ok=True)

base = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset"
info = pd.read_csv(f"{base}/M4-info.csv")

FREQ = "Daily"   # 너가 이미 Daily Finance로 했으니 일단 Daily 유지
H = 14          # M4 Daily horizon
finance_ids = info[(info["category"]=="Finance") & (info["SP"]==FREQ)]["M4id"].tolist()

train = pd.read_csv(f"{base}/Train/{FREQ}-train.csv").rename(columns={"V1":"unique_id"})
test  = pd.read_csv(f"{base}/Test/{FREQ}-test.csv").rename(columns={"V1":"unique_id"})

train = train[train["unique_id"].isin(finance_ids)].reset_index(drop=True)
test  = test[test["unique_id"].isin(finance_ids)].reset_index(drop=True)

print("Finance Daily:", train.shape, test.shape)

# wide -> long (unique_id, ds, y)
def wide_to_long(df_wide):
    df = df_wide.copy()
    vals = df.drop(columns=["unique_id"]).to_numpy()
    out = []
    for i, uid in enumerate(df["unique_id"].tolist()):
        y = vals[i]
        y = y[~np.isnan(y)]
        # ds는 단순 index로 두고, 모델에서 freq 의미만 맞추면 됨(일단 smoke)
        out.append(pd.DataFrame({"unique_id": uid, "ds": np.arange(len(y), dtype=int), "y": y.astype(np.float32)}))
    return pd.concat(out, axis=0, ignore_index=True)

df_train_long = wide_to_long(train)
# test는 미래 H만 있으니, 평가 때만 따로 사용(OWA 계산용)
df_train_long.to_parquet(PROJ/"data"/"m4_fin_daily_train_long.parquet", index=False)

(train[["unique_id"]]).to_csv(PROJ/"data"/"finance_daily_ids.csv", index=False)
test.to_parquet(PROJ/"data"/"m4_fin_daily_test_wide.parquet", index=False)

print("saved:",
      PROJ/"data"/"m4_fin_daily_train_long.parquet",
      PROJ/"data"/"m4_fin_daily_test_wide.parquet")


Finance Daily: (1559, 9920) (1559, 15)
saved: /content/m4_nbeats_timeo1/data/m4_fin_daily_train_long.parquet /content/m4_nbeats_timeo1/data/m4_fin_daily_test_wide.parquet


In [ ]:
# 셀 2) “loss 플러그인” 준비
import torch
import torch.nn.functional as F

def time_o1_loss(y_hat, y, alpha=0.5, gamma=0.5, eps=1e-6):
    """
    Time-o1 알고리즘(요지):
    - 정답 레이블 y를 표준화(평균0, 표준편차1)
    - y의 시간축 상관구조를 SVD로 뽑아(사실상 PCA)
    - 그 변환 공간에서 상위 K개 성분만 맞추도록 L1 손실 추가
    - 기존 시간영역 MSE(TMSE)와 가중합해서 학습 안정/자기상관 반영을 노림

    입력:
      y_hat, y: (배치, horizon)  단변량 예측값/정답
      alpha: 변환공간 손실 비중
      gamma: horizon에서 몇 개 성분(K)을 쓸지 비율
    """
    assert y_hat.shape == y.shape and y.dim() == 2, (y_hat.shape, y.shape)

    B, H = y.shape
    K = max(1, int(round(gamma * H)))

    # (1) y 기준으로 표준화(누수/불안정 방지: y_hat도 동일 mu/sigma로 표준화)
    mu = y.mean(dim=1, keepdim=True)
    sig = y.std(dim=1, keepdim=True).clamp_min(eps)
    y_std = (y - mu) / sig
    yhat_std = (y_hat - mu) / sig

    # (2) SVD로 변환 기저 P* 계산 (시간축 상관구조)
    _, _, Vh = torch.linalg.svd(y_std, full_matrices=False)
    P = Vh.transpose(-2, -1)  # (H, H)

    # (3) 변환 공간으로 투영
    Z = y_std @ P
    Zhat = yhat_std @ P

    # (4) 상위 K개 성분 L1 + 시간영역 MSE 결합
    l_trans = torch.abs(Zhat[:, :K] - Z[:, :K]).mean()
    l_tmse = F.mse_loss(y_hat, y)
    return alpha * l_trans + (1.0 - alpha) * l_tmse


In [ ]:
B, H = 32, 14
y = torch.randn(B, H)
y_hat = y + 0.1 * torch.randn(B, H)
loss = time_o1_loss(y_hat, y, alpha=0.5, gamma=0.5)
print(loss.item())


0.05014767125248909


In [ ]:
# 셀 3) Naive2(비교 기준) 예측/지표 저장 + 공통 metric 함수
import os, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJ = Path("/content/m4_nbeats_timeo1")
DATA = PROJ / "data"
ART  = PROJ / "artifacts"
LOG  = PROJ / "logs"
CKPT = PROJ / "checkpoints"
for p in [ART, LOG, CKPT]:
    os.makedirs(p, exist_ok=True)

FREQ = "Daily"
H = 14
m = 1  # M4 Daily는 seasonality=1로 처리

# 저장해둔 test_wide 사용
test = pd.read_parquet(DATA / "m4_fin_daily_test_wide.parquet")
ids = pd.read_csv(DATA / "finance_daily_ids.csv")["unique_id"].tolist()

# train은 wide 형태가 MASE scale 계산/Naive2에 편함 -> raw로 다시 받음(빠름)
base = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset"
train_wide = pd.read_csv(f"{base}/Train/{FREQ}-train.csv").rename(columns={"V1": "unique_id"})
train_wide = train_wide[train_wide["unique_id"].isin(ids)].reset_index(drop=True)

assert train_wide.shape[0] == 1559 and test.shape[0] == 1559, (train_wide.shape, test.shape)

def row_to_series(row):
    x = row.iloc[1:].to_numpy(dtype=np.float32)
    return x[~np.isnan(x)]

def smape_mean(y, yhat, eps=1e-8):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    num = np.abs(yhat - y)
    den = np.abs(y) + np.abs(yhat) + eps
    return float(200.0 * np.mean(num / den))

def mase_mean(insample_mat, y, yhat, m=1, eps=1e-8):
    # insample_mat: (N, T) NaN padding 가능
    insample_mat = np.asarray(insample_mat, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)

    diffs = np.abs(insample_mat[:, m:] - insample_mat[:, :-m])
    scale = np.nanmean(diffs, axis=1) + eps  # (N,)
    err = np.nanmean(np.abs(yhat - y), axis=1)  # (N,)
    return float(np.mean(err / scale))

def naive2_forecast(y_tr, h, m=1):
    y_tr = np.asarray(y_tr, dtype=np.float32)
    if y_tr.size == 0:
        return np.zeros(h, dtype=np.float32)
    if m > 1 and y_tr.size >= m:
        last = y_tr[-m:]
        reps = int(np.ceil(h / m))
        return np.tile(last, reps)[:h].astype(np.float32)
    return np.full(h, y_tr[-1], dtype=np.float32)

# y_true
y_true = test.drop(columns=["unique_id"]).to_numpy(dtype=np.float32)  # (1559, 14)

# insample matrix (MASE scale)
insample = train_wide.drop(columns=["unique_id"]).to_numpy(dtype=np.float32)  # (1559, T) + NaN

# naive2 preds
y_pred_n2 = np.zeros_like(y_true, dtype=np.float32)
for i, row in train_wide.iterrows():
    y_tr = row_to_series(row)
    y_pred_n2[i] = naive2_forecast(y_tr, H, m)

sm_n2 = smape_mean(y_true, y_pred_n2)
ma_n2 = mase_mean(insample, y_true, y_pred_n2, m=m)

naive2_metrics = {
    "dataset": "M4",
    "category": "Finance",
    "freq": FREQ,
    "N": int(y_true.shape[0]),
    "H": int(H),
    "m": int(m),
    "model": "Naive2",
    "sMAPE_mean": sm_n2,
    "MASE_mean": ma_n2,
    "OWA_vs_Naive2": 1.0,
}

np.savez_compressed(ART / "Daily_Finance_Naive2.npz",
                    ids=np.array(ids), y_true=y_true, y_pred=y_pred_n2)
(ART / "Daily_Finance_Naive2.json").write_text(json.dumps(naive2_metrics, indent=2), encoding="utf-8")

print(naive2_metrics)
print("saved:", ART / "Daily_Finance_Naive2.npz")
print("saved:", ART / "Daily_Finance_Naive2.json")


{'dataset': 'M4', 'category': 'Finance', 'freq': 'Daily', 'N': 1559, 'H': 14, 'm': 1, 'model': 'Naive2', 'sMAPE_mean': 3.517052389619388, 'MASE_mean': 3.5143761051316997, 'OWA_vs_Naive2': 1.0}
saved: /content/m4_nbeats_timeo1/artifacts/Daily_Finance_Naive2.npz
saved: /content/m4_nbeats_timeo1/artifacts/Daily_Finance_Naive2.json


In [ ]:
# 셀 4) N-BEATS 학습(1) : TMSE baseline (로그/체크포인트 저장)
'''
여기서부터가 “벤치마크-베이스라인”의 딥러닝 파트
우선 smoke로 돌리기 위해 steps를 적게 잡았음. (원하면 steps만 늘리면 전량 학습으로 확장됨)
'''
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJ = Path("/content/m4_nbeats_timeo1")
DATA = PROJ / "data"
LOG  = PROJ / "logs"
CKPT = PROJ / "checkpoints"
os.makedirs(LOG, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

# ---- 하이퍼파라미터(일단 안정적으로 돌아가는 값) ----
INPUT_SIZE = 256
H = 14
BATCH_SIZE = 256
LR = 1e-3
EPOCHS = 2
STEPS_PER_EPOCH = 200   # ✅ smoke. 2000~10000으로 키우면 학습이 “진짜” 됨.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ----------------------------------------------------

# train wide 다시 사용 (빠름)
base = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset"
train_wide = pd.read_csv(f"{base}/Train/Daily-train.csv").rename(columns={"V1":"unique_id"})
ids = pd.read_csv(DATA / "finance_daily_ids.csv")["unique_id"].tolist()
train_wide = train_wide[train_wide["unique_id"].isin(ids)].reset_index(drop=True)

def row_to_series(row):
    x = row.iloc[1:].to_numpy(dtype=np.float32)
    return x[~np.isnan(x)]

series_list = [row_to_series(r) for _, r in train_wide.iterrows()]
lengths = np.array([len(x) for x in series_list], dtype=np.int32)

def sample_batch(batch_size, input_size, h):
    xs = np.zeros((batch_size, input_size), dtype=np.float32)
    ys = np.zeros((batch_size, h), dtype=np.float32)
    for i in range(batch_size):
        sid = np.random.randint(0, len(series_list))
        y = series_list[sid]
        L = y.shape[0]
        # 최소 input+h 확보 안 되면 패딩으로 처리
        if L < input_size + h + 1:
            pad_val = y[-1] if L > 0 else 0.0
            y_pad = np.pad(y, (input_size + h + 1 - L, 0), mode="constant", constant_values=pad_val)
            y = y_pad
            L = y.shape[0]
        t = np.random.randint(input_size, L - h)  # 예측 시작점
        x_raw = y[t - input_size: t]
        y_raw = y[t: t + h]

        mu = x_raw.mean()
        sig = x_raw.std() + 1e-6

        xs[i] = (x_raw - mu) / sig
        ys[i] = (y_raw - mu) / sig
    return torch.from_numpy(xs), torch.from_numpy(ys)

class NBeatsBlock(nn.Module):
    def __init__(self, input_size, theta_size, hidden_size=512, n_layers=4):
        super().__init__()
        layers = []
        in_dim = input_size
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden_size))
            layers.append(nn.ReLU())
            in_dim = hidden_size
        self.fc = nn.Sequential(*layers)
        self.theta = nn.Linear(hidden_size, theta_size)

    def forward(self, x):
        h = self.fc(x)
        return self.theta(h)

class NBeats(nn.Module):
    def __init__(self, input_size, h, n_blocks=8, hidden_size=512, n_layers=4):
        super().__init__()
        self.input_size = input_size
        self.h = h
        self.blocks = nn.ModuleList([
            NBeatsBlock(input_size, theta_size=input_size + h, hidden_size=hidden_size, n_layers=n_layers)
            for _ in range(n_blocks)
        ])

    def forward(self, x):
        # x: (B, input_size)
        residual = x
        forecast = x.new_zeros((x.size(0), self.h))
        for blk in self.blocks:
            theta = blk(residual)
            backcast = theta[:, : self.input_size]
            fcst = theta[:, self.input_size :]
            residual = residual - backcast
            forecast = forecast + fcst
        return forecast

# TMSE loss
def tmse_loss(y_hat, y):
    return F.mse_loss(y_hat, y)

# 로깅
log_path = LOG / "train_nbeats_tmse.log"
def log(msg):
    s = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(s)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(s + "\n")

torch.manual_seed(42)
np.random.seed(42)

model = NBeats(INPUT_SIZE, H, n_blocks=8, hidden_size=512, n_layers=4).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
from torch.amp import GradScaler, autocast
use_amp = (DEVICE == "cuda")
scaler = GradScaler(enabled=use_amp)


log(f"DEVICE={DEVICE} N={len(series_list)} INPUT_SIZE={INPUT_SIZE} H={H} BATCH={BATCH_SIZE} EPOCHS={EPOCHS} STEPS_PER_EPOCH={STEPS_PER_EPOCH}")

model.train()
for ep in range(1, EPOCHS + 1):
    losses = []
    t0 = time.time()
    for step in range(1, STEPS_PER_EPOCH + 1):
        x, y = sample_batch(BATCH_SIZE, INPUT_SIZE, H)
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        opt.zero_grad(set_to_none=True)
        with autocast(device_type="cuda", enabled=use_amp):
            y_hat = model(x)
            loss = tmse_loss(y_hat, y)


        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        losses.append(loss.item())

        if step % 50 == 0:
            log(f"ep={ep} step={step}/{STEPS_PER_EPOCH} loss={np.mean(losses[-50:]):.6f}")

    log(f"ep={ep} done avg_loss={np.mean(losses):.6f} time={time.time()-t0:.1f}s")

ckpt_path = CKPT / "nbeats_tmse.pt"
torch.save({"model": model.state_dict(),
            "input_size": INPUT_SIZE,
            "h": H}, ckpt_path)
log(f"saved checkpoint: {ckpt_path}")

# tail 출력
print("\n--- log tail ---")
print("\n".join(open(log_path, "r", encoding="utf-8").read().splitlines()[-20:]))


[2026-01-16 14:21:44] DEVICE=cuda N=1559 INPUT_SIZE=256 H=14 BATCH=256 EPOCHS=2 STEPS_PER_EPOCH=200
[2026-01-16 14:21:46] ep=1 step=50/200 loss=408.381910
[2026-01-16 14:21:48] ep=1 step=100/200 loss=574.690357
[2026-01-16 14:21:49] ep=1 step=150/200 loss=0.379038
[2026-01-16 14:21:51] ep=1 step=200/200 loss=293.245522
[2026-01-16 14:21:51] ep=1 done avg_loss=319.174206 time=6.6s
[2026-01-16 14:21:52] ep=2 step=50/200 loss=0.404322
[2026-01-16 14:21:54] ep=2 step=100/200 loss=0.323029
[2026-01-16 14:21:55] ep=2 step=150/200 loss=269.596986
[2026-01-16 14:21:57] ep=2 step=200/200 loss=25.739339
[2026-01-16 14:21:57] ep=2 done avg_loss=74.015919 time=6.2s
[2026-01-16 14:21:57] saved checkpoint: /content/m4_nbeats_timeo1/checkpoints/nbeats_tmse.pt

--- log tail ---
[2026-01-16 14:21:44] DEVICE=cuda N=1559 INPUT_SIZE=256 H=14 BATCH=256 EPOCHS=2 STEPS_PER_EPOCH=200
[2026-01-16 14:21:46] ep=1 step=50/200 loss=408.381910
[2026-01-16 14:21:48] ep=1 step=100/200 loss=574.690357
[2026-01-16 14:2

In [ ]:
# 셀 5) N-BEATS 평가 + OWA 계산 + 예측 저장 (TMSE 버전)
import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

PROJ = Path("/content/m4_nbeats_timeo1")
DATA = PROJ / "data"
ART  = PROJ / "artifacts"
LOG  = PROJ / "logs"
CKPT = PROJ / "checkpoints"
for p in [ART, LOG, CKPT]:
    os.makedirs(p, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 공통 metric 로드
naive2_metrics = json.loads((ART / "Daily_Finance_Naive2.json").read_text(encoding="utf-8"))
sm_n2 = naive2_metrics["sMAPE_mean"]
ma_n2 = naive2_metrics["MASE_mean"]

test = pd.read_parquet(DATA / "m4_fin_daily_test_wide.parquet")
ids = pd.read_csv(DATA / "finance_daily_ids.csv")["unique_id"].tolist()

base = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset"
train_wide = pd.read_csv(f"{base}/Train/Daily-train.csv").rename(columns={"V1":"unique_id"})
train_wide = train_wide[train_wide["unique_id"].isin(ids)].reset_index(drop=True)

def row_to_series(row):
    x = row.iloc[1:].to_numpy(dtype=np.float32)
    return x[~np.isnan(x)]

def smape_mean(y, yhat, eps=1e-8):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    num = np.abs(yhat - y)
    den = np.abs(y) + np.abs(yhat) + eps
    return float(200.0 * np.mean(num / den))

def mase_mean(insample_mat, y, yhat, m=1, eps=1e-8):
    insample_mat = np.asarray(insample_mat, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    diffs = np.abs(insample_mat[:, m:] - insample_mat[:, :-m])
    scale = np.nanmean(diffs, axis=1) + eps
    err = np.nanmean(np.abs(yhat - y), axis=1)
    return float(np.mean(err / scale))

# 모델 정의는 학습 셀과 동일해야 함
import torch.nn as nn

class NBeatsBlock(nn.Module):
    def __init__(self, input_size, theta_size, hidden_size=512, n_layers=4):
        super().__init__()
        layers = []
        in_dim = input_size
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden_size))
            layers.append(nn.ReLU())
            in_dim = hidden_size
        self.fc = nn.Sequential(*layers)
        self.theta = nn.Linear(hidden_size, theta_size)

    def forward(self, x):
        h = self.fc(x)
        return self.theta(h)

class NBeats(nn.Module):
    def __init__(self, input_size, h, n_blocks=8, hidden_size=512, n_layers=4):
        super().__init__()
        self.input_size = input_size
        self.h = h
        self.blocks = nn.ModuleList([
            NBeatsBlock(input_size, theta_size=input_size + h, hidden_size=hidden_size, n_layers=n_layers)
            for _ in range(n_blocks)
        ])

    def forward(self, x):
        residual = x
        forecast = x.new_zeros((x.size(0), self.h))
        for blk in self.blocks:
            theta = blk(residual)
            backcast = theta[:, : self.input_size]
            fcst = theta[:, self.input_size :]
            residual = residual - backcast
            forecast = forecast + fcst
        return forecast

# 체크포인트 로드
ckpt = torch.load(CKPT / "nbeats_tmse.pt", map_location="cpu")
INPUT_SIZE = int(ckpt["input_size"])
H = int(ckpt["h"])

model = NBeats(INPUT_SIZE, H, n_blocks=8, hidden_size=512, n_layers=4).to(DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

# 예측 생성: 각 시계열의 마지막 INPUT_SIZE로 H-step 예측
y_true = test.drop(columns=["unique_id"]).to_numpy(dtype=np.float32)  # (N, H)
y_pred = np.zeros_like(y_true, dtype=np.float32)

with torch.no_grad():
    for i, row in train_wide.iterrows():
        y = row_to_series(row)
        if y.size < INPUT_SIZE:
            pad_val = y[-1] if y.size > 0 else 0.0
            y = np.pad(y, (INPUT_SIZE - y.size, 0), mode="constant", constant_values=pad_val)
        x_raw = y[-INPUT_SIZE:].astype(np.float32)
        mu = float(x_raw.mean())
        sig = float(x_raw.std() + 1e-6)

        x = torch.from_numpy(((x_raw - mu) / sig))[None, :].to(DEVICE)
        out_norm = model(x).squeeze(0).detach().cpu().numpy().astype(np.float32)

        y_pred[i] = out_norm * sig + mu


# MASE용 insample
insample = train_wide.drop(columns=["unique_id"]).to_numpy(dtype=np.float32)

sm = smape_mean(y_true, y_pred)
ma = mase_mean(insample, y_true, y_pred, m=1)

owa = 0.5 * ((sm / sm_n2) + (ma / ma_n2))

metrics = {
    "dataset": "M4",
    "category": "Finance",
    "freq": "Daily",
    "N": int(y_true.shape[0]),
    "H": int(H),
    "m": 1,
    "model": "NBEATS_TMSE",
    "sMAPE_mean": sm,
    "MASE_mean": ma,
    "OWA_vs_Naive2": float(owa),
    "naive2_ref": {"sMAPE_mean": sm_n2, "MASE_mean": ma_n2},
}

np.savez_compressed(ART / "Daily_Finance_NBEATS_TMSE.npz",
                    ids=np.array(ids), y_true=y_true, y_pred=y_pred)
(ART / "Daily_Finance_NBEATS_TMSE.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(metrics)
print("saved:", ART / "Daily_Finance_NBEATS_TMSE.npz")
print("saved:", ART / "Daily_Finance_NBEATS_TMSE.json")


{'dataset': 'M4', 'category': 'Finance', 'freq': 'Daily', 'N': 1559, 'H': 14, 'm': 1, 'model': 'NBEATS_TMSE', 'sMAPE_mean': 4.126332146029407, 'MASE_mean': 3.8801741947186272, 'OWA_vs_Naive2': 1.1386610579337892, 'naive2_ref': {'sMAPE_mean': 3.517052389619388, 'MASE_mean': 3.5143761051316997}}
saved: /content/m4_nbeats_timeo1/artifacts/Daily_Finance_NBEATS_TMSE.npz
saved: /content/m4_nbeats_timeo1/artifacts/Daily_Finance_NBEATS_TMSE.json


In [ ]:
import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# sample_batch / model / time_o1_loss 가 이미 정의돼 있다고 가정
x, y = sample_batch(256, INPUT_SIZE, H)
x = x.to(DEVICE)
y = y.to(DEVICE)

model.train()
with torch.no_grad():
    y_hat = model(x)
    # 핵심: loss는 float32로 계산 (SVD 안정화)
    loss32 = time_o1_loss(y_hat.float(), y.float(), alpha=0.5, gamma=0.5, eps=1e-4)

print("loss32:", float(loss32))
print("isfinite:", torch.isfinite(loss32).item())


loss32: 1.7713313102722168
isfinite: True


In [ ]:
import math
import torch
import torch.nn.functional as F

def make_dct_mat(H: int, device: str):
    n = torch.arange(H, device=device).float()
    k = torch.arange(H, device=device).float().unsqueeze(1)
    mat = torch.cos(math.pi / H * (n + 0.5) * k)  # (H,H)
    mat[0] *= math.sqrt(1.0 / H)
    if H > 1:
        mat[1:] *= math.sqrt(2.0 / H)
    return mat  # (H,H)

class TimeO1DCTLoss(torch.nn.Module):
    """
    sample_batch에서 이미 window 정규화를 했다는 전제.
    => 여기서는 추가 표준화 없이 변환공간 정렬 + TMSE 혼합만 적용.
    """
    def __init__(self, H: int, alpha: float = 0.2, gamma: float = 0.5):
        super().__init__()
        self.H = H
        self.alpha = alpha
        self.gamma = gamma
        self.register_buffer("P", make_dct_mat(H, device="cpu"), persistent=False)

    def forward(self, y_hat, y):
        assert y_hat.shape == y.shape and y.dim() == 2
        B, H = y.shape
        K = max(1, int(round(self.gamma * H)))

        P = self.P.to(y.device)

        Z = y.float() @ P.T
        Zhat = y_hat.float() @ P.T

        l_trans = torch.abs(Zhat[:, :K] - Z[:, :K]).mean()
        l_tmse = F.mse_loss(y_hat.float(), y.float())
        return self.alpha * l_trans + (1.0 - self.alpha) * l_tmse

print("TimeO1DCTLoss defined")


TimeO1DCTLoss defined


In [ ]:
# 셀 6) N-BEATS 학습(2): Time-o1 loss 버전 (로그/체크포인트 저장)
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJ = Path("/content/m4_nbeats_timeo1")
DATA = PROJ / "data"
LOG  = PROJ / "logs"
CKPT = PROJ / "checkpoints"
os.makedirs(LOG, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

# ---- 방어: time_o1_loss 없으면 정의 ----
if "time_o1_loss" not in globals():
    def time_o1_loss(y_hat, y, alpha=0.5, gamma=0.5, eps=1e-6):
        assert y_hat.shape == y.shape and y.dim() == 2, (y_hat.shape, y.shape)
        B, H = y.shape
        K = max(1, int(round(gamma * H)))

        mu = y.mean(dim=1, keepdim=True)
        sig = y.std(dim=1, keepdim=True).clamp_min(eps)
        y_std = (y - mu) / sig
        yhat_std = (y_hat - mu) / sig

        _, _, Vh = torch.linalg.svd(y_std.float(), full_matrices=False)
        P = Vh.transpose(-2, -1)  # (H,H)

        Z = y_std @ P
        Zhat = yhat_std @ P

        l_trans = torch.abs(Zhat[:, :K] - Z[:, :K]).mean()
        l_tmse = F.mse_loss(y_hat, y)
        return alpha * l_trans + (1.0 - alpha) * l_tmse

# ---- 방어: 모델 클래스 없으면 정의 ----
if "NBeats" not in globals():
    class NBeatsBlock(nn.Module):
        def __init__(self, input_size, theta_size, hidden_size=512, n_layers=4):
            super().__init__()
            layers = []
            in_dim = input_size
            for _ in range(n_layers):
                layers.append(nn.Linear(in_dim, hidden_size))
                layers.append(nn.ReLU())
                in_dim = hidden_size
            self.fc = nn.Sequential(*layers)
            self.theta = nn.Linear(hidden_size, theta_size)

        def forward(self, x):
            h = self.fc(x)
            return self.theta(h)

    class NBeats(nn.Module):
        def __init__(self, input_size, h, n_blocks=8, hidden_size=512, n_layers=4):
            super().__init__()
            self.input_size = input_size
            self.h = h
            self.blocks = nn.ModuleList([
                NBeatsBlock(input_size, theta_size=input_size + h, hidden_size=hidden_size, n_layers=n_layers)
                for _ in range(n_blocks)
            ])

        def forward(self, x):
            residual = x
            forecast = x.new_zeros((x.size(0), self.h))
            for blk in self.blocks:
                theta = blk(residual)
                backcast = theta[:, : self.input_size]
                fcst = theta[:, self.input_size :]
                residual = residual - backcast
                forecast = forecast + fcst
            return forecast

# ---- 하이퍼파라미터(우선 smoke) ----
INPUT_SIZE = 256
H = 14
BATCH_SIZE = 256
LR = 1e-3
EPOCHS = 3
STEPS_PER_EPOCH = 2000   # smoke. 안정화되면 2000~10000으로 키우면 됨
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ALPHA = 0.5
GAMMA = 0.5
# -----------------------------------

# train wide 다시 로드 (finance 1559)
base = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset"
ids = pd.read_csv(DATA / "finance_daily_ids.csv")["unique_id"].tolist()
train_wide = pd.read_csv(f"{base}/Train/Daily-train.csv").rename(columns={"V1":"unique_id"})
train_wide = train_wide[train_wide["unique_id"].isin(ids)].reset_index(drop=True)

def row_to_series(row):
    x = row.iloc[1:].to_numpy(dtype=np.float32)
    return x[~np.isnan(x)]

series_list = [row_to_series(r) for _, r in train_wide.iterrows()]

def sample_batch(batch_size, input_size, h):
    xs = np.zeros((batch_size, input_size), dtype=np.float32)
    ys = np.zeros((batch_size, h), dtype=np.float32)
    for i in range(batch_size):
        sid = np.random.randint(0, len(series_list))
        y = series_list[sid]
        L = y.shape[0]
        if L < input_size + h + 1:
            pad_val = y[-1] if L > 0 else 0.0
            y = np.pad(y, (input_size + h + 1 - L, 0), mode="constant", constant_values=pad_val)
            L = y.shape[0]
        t = np.random.randint(input_size, L - h)
        x_raw = y[t - input_size: t]
        y_raw = y[t: t + h]

        mu = x_raw.mean()
        sig = x_raw.std() + 1e-6

        xs[i] = (x_raw - mu) / sig
        ys[i] = (y_raw - mu) / sig
    return torch.from_numpy(xs), torch.from_numpy(ys)

# 로깅
log_path = LOG / f"train_nbeats_timeo1_a{ALPHA}_g{GAMMA}.log"
open(log_path, "w", encoding="utf-8").close()

def log(msg):
    s = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(s)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(s + "\n")

torch.manual_seed(42)
np.random.seed(42)

model = NBeats(INPUT_SIZE, H, n_blocks=8, hidden_size=512, n_layers=4).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = TimeO1DCTLoss(H=H, alpha=0.2, gamma=0.5).to(DEVICE)
from torch.amp import GradScaler, autocast
use_amp = (DEVICE == "cuda")
scaler = GradScaler(enabled=use_amp)


log(f"DEVICE={DEVICE} N={len(series_list)} INPUT_SIZE={INPUT_SIZE} H={H} BATCH={BATCH_SIZE} EPOCHS={EPOCHS} STEPS_PER_EPOCH={STEPS_PER_EPOCH} alpha={ALPHA} gamma={GAMMA}")

model.train()
for ep in range(1, EPOCHS + 1):
    losses = []
    t0 = time.time()
    for step in range(1, STEPS_PER_EPOCH + 1):
        x, y = sample_batch(BATCH_SIZE, INPUT_SIZE, H)
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        opt.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", enabled=use_amp):
            y_hat = model(x)

        loss = loss_fn(y_hat, y)

        if not torch.isfinite(loss):
            log("loss is NaN/Inf -> skip step")
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()


        losses.append(loss.item())
        if step % 50 == 0:
            log(f"ep={ep} step={step}/{STEPS_PER_EPOCH} loss={np.mean(losses[-50:]):.6f}")

    log(f"ep={ep} done avg_loss={np.mean(losses):.6f} time={time.time()-t0:.1f}s")

ckpt_path = CKPT / f"nbeats_timeo1_a{ALPHA}_g{GAMMA}.pt"
torch.save({"model": model.state_dict(), "input_size": INPUT_SIZE, "h": H, "alpha": ALPHA, "gamma": GAMMA}, ckpt_path)
log(f"saved checkpoint: {ckpt_path}")

print("\n--- log tail ---")
print("\n".join(open(log_path, "r", encoding="utf-8").read().splitlines()[-20:]))

[2026-01-16 14:36:54] DEVICE=cuda N=1559 INPUT_SIZE=256 H=14 BATCH=256 EPOCHS=3 STEPS_PER_EPOCH=2000 alpha=0.5 gamma=0.5
[2026-01-16 14:36:56] ep=1 step=50/2000 loss=326.846285
[2026-01-16 14:36:58] ep=1 step=100/2000 loss=459.886244
[2026-01-16 14:36:59] ep=1 step=150/2000 loss=0.386623
[2026-01-16 14:37:01] ep=1 step=200/2000 loss=234.646489
[2026-01-16 14:37:02] ep=1 step=250/2000 loss=0.405557
[2026-01-16 14:37:04] ep=1 step=300/2000 loss=0.339973
[2026-01-16 14:37:05] ep=1 step=350/2000 loss=215.617538
[2026-01-16 14:37:07] ep=1 step=400/2000 loss=20.730296
[2026-01-16 14:37:09] ep=1 step=450/2000 loss=106.162368
[2026-01-16 14:37:11] ep=1 step=500/2000 loss=5.666216
[2026-01-16 14:37:12] ep=1 step=550/2000 loss=356.242553
[2026-01-16 14:37:13] ep=1 step=600/2000 loss=177.325192
[2026-01-16 14:37:15] ep=1 step=650/2000 loss=105.870066
[2026-01-16 14:37:16] ep=1 step=700/2000 loss=618.384970
[2026-01-16 14:37:18] ep=1 step=750/2000 loss=392.001597
[2026-01-16 14:37:20] ep=1 step=80

In [ ]:
# 셀 7 평가 + OWA 계산 + 예측 저장 (Time-o1 loss 버전)
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJ = Path("/content/m4_nbeats_timeo1")
DATA = PROJ / "data"
ART  = PROJ / "artifacts"
CKPT = PROJ / "checkpoints"
os.makedirs(ART, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Naive2 기준 지표 로드
naive2 = json.loads((ART / "Daily_Finance_Naive2.json").read_text(encoding="utf-8"))
sm_n2 = naive2["sMAPE_mean"]
ma_n2 = naive2["MASE_mean"]

# 데이터 로드
test = pd.read_parquet(DATA / "m4_fin_daily_test_wide.parquet")
ids = pd.read_csv(DATA / "finance_daily_ids.csv")["unique_id"].tolist()

base = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset"
train_wide = pd.read_csv(f"{base}/Train/Daily-train.csv").rename(columns={"V1":"unique_id"})
train_wide = train_wide[train_wide["unique_id"].isin(ids)].reset_index(drop=True)

def row_to_series(row):
    x = row.iloc[1:].to_numpy(dtype=np.float32)
    return x[~np.isnan(x)]

def smape_mean(y, yhat, eps=1e-8):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    num = np.abs(yhat - y)
    den = np.abs(y) + np.abs(yhat) + eps
    return float(200.0 * np.mean(num / den))

def mase_mean(insample_mat, y, yhat, m=1, eps=1e-8):
    insample_mat = np.asarray(insample_mat, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    diffs = np.abs(insample_mat[:, m:] - insample_mat[:, :-m])
    scale = np.nanmean(diffs, axis=1) + eps
    err = np.nanmean(np.abs(yhat - y), axis=1)
    return float(np.mean(err / scale))

# 모델(학습 셀과 동일)
class NBeatsBlock(nn.Module):
    def __init__(self, input_size, theta_size, hidden_size=512, n_layers=4):
        super().__init__()
        layers = []
        in_dim = input_size
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden_size))
            layers.append(nn.ReLU())
            in_dim = hidden_size
        self.fc = nn.Sequential(*layers)
        self.theta = nn.Linear(hidden_size, theta_size)

    def forward(self, x):
        h = self.fc(x)
        return self.theta(h)

class NBeats(nn.Module):
    def __init__(self, input_size, h, n_blocks=8, hidden_size=512, n_layers=4):
        super().__init__()
        self.input_size = input_size
        self.h = h
        self.blocks = nn.ModuleList([
            NBeatsBlock(input_size, theta_size=input_size + h, hidden_size=hidden_size, n_layers=n_layers)
            for _ in range(n_blocks)
        ])

    def forward(self, x):
        residual = x
        forecast = x.new_zeros((x.size(0), self.h))
        for blk in self.blocks:
            theta = blk(residual)
            backcast = theta[:, : self.input_size]
            fcst = theta[:, self.input_size :]
            residual = residual - backcast
            forecast = forecast + fcst
        return forecast

# ckpt 로드
ckpt_path = CKPT / "nbeats_timeo1_a0.5_g0.5.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")
INPUT_SIZE = int(ckpt["input_size"])
H = int(ckpt["h"])
ALPHA = float(ckpt.get("alpha", 0.5))
GAMMA = float(ckpt.get("gamma", 0.5))

model = NBeats(INPUT_SIZE, H, n_blocks=8, hidden_size=512, n_layers=4).to(DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

# 예측
y_true = test.drop(columns=["unique_id"]).to_numpy(dtype=np.float32)  # (N,H)
y_pred = np.zeros_like(y_true, dtype=np.float32)

with torch.no_grad():
    for i, row in train_wide.iterrows():
        y = row_to_series(row)
        if y.size < INPUT_SIZE:
            pad_val = y[-1] if y.size > 0 else 0.0
            y = np.pad(y, (INPUT_SIZE - y.size, 0), mode="constant", constant_values=pad_val)
        x_raw = y[-INPUT_SIZE:].astype(np.float32)
        mu = float(x_raw.mean())
        sig = float(x_raw.std() + 1e-6)

        x = torch.from_numpy(((x_raw - mu) / sig))[None, :].to(DEVICE)
        out_norm = model(x).squeeze(0).detach().cpu().numpy().astype(np.float32)

        y_pred[i] = out_norm * sig + mu


# 지표/OWA
insample = train_wide.drop(columns=["unique_id"]).to_numpy(dtype=np.float32)

sm = smape_mean(y_true, y_pred)
ma = mase_mean(insample, y_true, y_pred, m=1)
owa = 0.5 * ((sm / sm_n2) + (ma / ma_n2))

metrics = {
    "dataset": "M4",
    "category": "Finance",
    "freq": "Daily",
    "N": int(y_true.shape[0]),
    "H": int(H),
    "m": 1,
    "model": "NBEATS_TIMEO1",
    "alpha": ALPHA,
    "gamma": GAMMA,
    "sMAPE_mean": sm,
    "MASE_mean": ma,
    "OWA_vs_Naive2": float(owa),
    "naive2_ref": {"sMAPE_mean": sm_n2, "MASE_mean": ma_n2},
}

np.savez_compressed(ART / "Daily_Finance_NBEATS_TIMEO1.npz",
                    ids=np.array(ids), y_true=y_true, y_pred=y_pred)
(ART / "Daily_Finance_NBEATS_TIMEO1.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(metrics)
print("saved:", ART / "Daily_Finance_NBEATS_TIMEO1.npz")
print("saved:", ART / "Daily_Finance_NBEATS_TIMEO1.json")


{'dataset': 'M4', 'category': 'Finance', 'freq': 'Daily', 'N': 1559, 'H': 14, 'm': 1, 'model': 'NBEATS_TIMEO1', 'alpha': 0.5, 'gamma': 0.5, 'sMAPE_mean': 3.95705801783048, 'MASE_mean': 3.738559013040475, 'OWA_vs_Naive2': 1.0944483026354952, 'naive2_ref': {'sMAPE_mean': 3.517052389619388, 'MASE_mean': 3.5143761051316997}}
saved: /content/m4_nbeats_timeo1/artifacts/Daily_Finance_NBEATS_TIMEO1.npz
saved: /content/m4_nbeats_timeo1/artifacts/Daily_Finance_NBEATS_TIMEO1.json


In [ ]:
# 셀 8 TMSE vs Time-o1 한 줄 비교
import json
from pathlib import Path

ART = Path("/content/m4_nbeats_timeo1/artifacts")

tmse = json.loads((ART / "Daily_Finance_NBEATS_TMSE.json").read_text(encoding="utf-8"))
t1   = json.loads((ART / "Daily_Finance_NBEATS_TIMEO1.json").read_text(encoding="utf-8"))

print("TMSE  OWA:", tmse["OWA_vs_Naive2"], "sMAPE:", tmse["sMAPE_mean"], "MASE:", tmse["MASE_mean"])
print("T-o1 OWA:", t1["OWA_vs_Naive2"],  "sMAPE:", t1["sMAPE_mean"],  "MASE:", t1["MASE_mean"])


TMSE  OWA: 1.1386610579337892 sMAPE: 4.126332146029407 MASE: 3.8801741947186272
T-o1 OWA: 1.0944483026354952 sMAPE: 3.95705801783048 MASE: 3.738559013040475


In [ ]:
# 검증 셀
import numpy as np
import pandas as pd

base = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset"
train_wide = pd.read_csv(f"{base}/Train/Daily-train.csv").rename(columns={"V1":"unique_id"})

# finance id만
info = pd.read_csv(f"{base}/M4-info.csv")
finance_ids = info[(info["category"]=="Finance") & (info["SP"]=="Daily")]["M4id"].tolist()
train_wide = train_wide[train_wide["unique_id"].isin(finance_ids)].reset_index(drop=True)

vals = train_wide.drop(columns=["unique_id"]).to_numpy(dtype=float)

# 시계열별 표준편차(대략적인 스케일)
stds = np.nanstd(vals, axis=1)
qs = np.quantile(stds, [0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0])
print("series std quantiles:", qs)


series std quantiles: [   30.53661421   458.12245591  2307.92705692  3334.70536214
  3930.15566021  5551.69334646 14543.48656578]


In [ ]:
# 셀 9) Time-o1 평가(OWA) 재실행 + 저장파일 확인
import json
from pathlib import Path

ART = Path("/content/m4_nbeats_timeo1/artifacts")
CKPT = Path("/content/m4_nbeats_timeo1/checkpoints")

print("ckpt exists:", (CKPT / "nbeats_timeo1_a0.5_g0.5.pt").exists())

# 셀7 다시 실행했다고 가정하고 결과 비교
tmse = json.loads((ART / "Daily_Finance_NBEATS_TMSE.json").read_text(encoding="utf-8"))
t1   = json.loads((ART / "Daily_Finance_NBEATS_TIMEO1.json").read_text(encoding="utf-8"))

print("TMSE :", tmse["OWA_vs_Naive2"], tmse["sMAPE_mean"], tmse["MASE_mean"])
print("T-o1:", t1["OWA_vs_Naive2"],  t1["sMAPE_mean"],  t1["MASE_mean"])
print("saved:", (ART / "Daily_Finance_NBEATS_TIMEO1.json"))
print("saved:", (ART / "Daily_Finance_NBEATS_TIMEO1.npz"))


ckpt exists: True
TMSE : 1.1386610579337892 4.126332146029407 3.8801741947186272
T-o1: 1.0944483026354952 3.95705801783048 3.738559013040475
saved: /content/m4_nbeats_timeo1/artifacts/Daily_Finance_NBEATS_TIMEO1.json
saved: /content/m4_nbeats_timeo1/artifacts/Daily_Finance_NBEATS_TIMEO1.npz


In [ ]:
# 셀 10) Time-o1 loss를 “고정 DCT 변환” 버전으로 정의 (SVD 제거)
import math
import torch
import torch.nn.functional as F

def make_dct_mat(H: int, device: str):
    # DCT-II (정규화 포함). HxH
    n = torch.arange(H, device=device).float()
    k = torch.arange(H, device=device).float().unsqueeze(1)
    mat = torch.cos(math.pi / H * (n + 0.5) * k)  # (H,H)

    # orthonormal scaling
    mat[0] *= math.sqrt(1.0 / H)
    if H > 1:
        mat[1:] *= math.sqrt(2.0 / H)
    return mat  # (H,H)

class TimeO1DCTLoss(torch.nn.Module):
    """
    너는 이미 sample_batch에서 window 정규화를 했으니까,
    여기서는 '추가 표준화'를 하지 않고 변환공간 정렬만 적용.
    """
    def __init__(self, H: int, alpha: float = 0.2, gamma: float = 0.5):
        super().__init__()
        self.H = H
        self.alpha = alpha
        self.gamma = gamma
        self.register_buffer("P", make_dct_mat(H, device="cpu"), persistent=False)

    def forward(self, y_hat, y):
        # y_hat, y: (B,H)  (둘 다 window 정규화된 값)
        assert y_hat.shape == y.shape and y.dim() == 2
        B, H = y.shape
        K = max(1, int(round(self.gamma * H)))

        # 변환행렬 device 맞추기
        P = self.P.to(y.device)

        # 변환공간 (B,H)
        Z = y.float() @ P.T
        Zhat = y_hat.float() @ P.T

        l_trans = torch.abs(Zhat[:, :K] - Z[:, :K]).mean()
        l_tmse = F.mse_loss(y_hat.float(), y.float())
        return self.alpha * l_trans + (1.0 - self.alpha) * l_tmse

print("TimeO1DCTLoss ready")


TimeO1DCTLoss ready
